# DSCI552: Final Project
- Name: Andrea Fernandez Cruz
- Github Username: ndreaelizabth
- USC ID: 6735339003


In [1]:
from pathlib import Path
import os
import random
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Setup imports complete.")
print("Current folder:", Path.cwd())

Setup imports complete.
Current folder: /Users/ndreaelizabth/Desktop/dsci552/dsci552_finalproject_spring_2026/notebook


In [3]:
PROJECT_ROOT = Path("..")

# Your dataset may be either data/C1...C5 or data/defungi/C1...C5
if (PROJECT_ROOT / "data" / "defungi").exists():
    DATA_DIR = PROJECT_ROOT / "data" / "defungi"
else:
    DATA_DIR = PROJECT_ROOT / "data"

SAVED_MODELS_DIR = Path("saved_models")
PLOTS_DIR = Path("plots")
RESULTS_DIR = Path("results")

SAVED_MODELS_DIR.mkdir(exist_ok=True)
PLOTS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

classes = sorted([folder.name for folder in DATA_DIR.iterdir() if folder.is_dir()])

print("Classes found:", classes)
print("Number of classes:", len(classes))

Classes found: ['C1', 'C2', 'C3', 'C4', 'C5']
Number of classes: 5


In [4]:
image_counts = {}

for class_name in classes:
    class_folder = DATA_DIR / class_name
    
    image_files = [
        file for file in class_folder.iterdir()
        if file.suffix.lower() in [".jpg", ".jpeg", ".png"]
    ]
    
    image_counts[class_name] = len(image_files)

image_counts_df = pd.DataFrame.from_dict(
    image_counts,
    orient="index",
    columns=["Number of Images"]
)

image_counts_df

,Number of Images
C1,4404
C2,2334
C3,819
C4,818
C5,739


In [5]:
SPLIT_DIR = PROJECT_ROOT / "data" / "split"

TRAIN_DIR = SPLIT_DIR / "train"
VAL_DIR = SPLIT_DIR / "val"
TEST_DIR = SPLIT_DIR / "test"

print("Split folder will be:", SPLIT_DIR)

Split folder will be: ../data/split


In [7]:
def split_dataset(source_dir, train_dir, val_dir, test_dir, seed=42):
    random.seed(seed)

    train_dir.mkdir(parents=True, exist_ok=True)
    val_dir.mkdir(parents=True, exist_ok=True)
    test_dir.mkdir(parents=True, exist_ok=True)

    for class_name in classes:
        class_path = source_dir / class_name

        image_paths = [
            file for file in class_path.iterdir()
            if file.suffix.lower() in [".jpg", ".jpeg", ".png"]
        ]

        random.shuffle(image_paths)

        total = len(image_paths)

        train_end = int(total * 0.75)
        val_end = train_end + int(total * 0.15)

        train_images = image_paths[:train_end]
        val_images = image_paths[train_end:val_end]
        test_images = image_paths[val_end:]

        split_info = [
            (train_images, train_dir),
            (val_images, val_dir),
            (test_images, test_dir)
        ]

        for images, split_folder in split_info:
            class_split_folder = split_folder / class_name
            class_split_folder.mkdir(parents=True, exist_ok=True)

            for image_path in images:
                destination = class_split_folder / image_path.name

                if not destination.exists():
                    shutil.copy(image_path, destination)

        print(
            f"{class_name}: "
            f"train={len(train_images)}, "
            f"validation={len(val_images)}, "
            f"test={len(test_images)}, "
            f"total={total}"
        )

In [10]:
split_dataset(DATA_DIR, TRAIN_DIR, VAL_DIR, TEST_DIR)

C1: train=3303, validation=660, test=441, total=4404
C2: train=1750, validation=350, test=234, total=2334
C3: train=614, validation=122, test=83, total=819
C4: train=613, validation=122, test=83, total=818
C5: train=554, validation=110, test=75, total=739


In [9]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

print("TensorFlow version:", tf.__version__)
print("Keras is working.")

gpus = tf.config.list_physical_devices("GPU")
print("GPUs found:", gpus)

ModuleNotFoundError: No module named 'tensorflow'